# Notebook 04 — Stage 1A: canonical schema and presence flags

**Role in the pipeline:** first transformation stage on the canonical
data critical path. Reads the QC-corrected `derived.csv` produced by
Notebook 02 and produces two analysis-ready frames plus an audit trail.

```text
02_ta_ph_qc.ipynb
   └── sheet_<x>/data/derived.csv
                     │
                     ▼
          04_stage1a.ipynb         ◄── THIS NOTEBOOK
            • resolve canonical column names (alias resolution)
            • normalise TA units and pH scale labels
            • flag out-of-range values (advisory, never destructive)
            • flag possible duplicates
            • write staged.csv + analysis_ready.csv (+ Parquet)
                              │
                              ▼
                  05_stage1b.ipynb  →  06_stage2  →  ...
```

**What "Stage 1A" produces**

- `staged.csv` — every column the input had, *plus* the canonical columns
  copied from their aliases, *plus* the flag columns. **This is the Stage 1B
  input used by the current runner.**
- `analysis_ready.csv` — same rows, but columns reordered into the
  canonical export order. Use this for review and downstream export.
- Audit logs in `logs/`: rename audit, canonical inventory,
  missingness inventory, effective config, and `manifest.json`.
- A one-page `report.md`.

See `04_stage1a.README.md` for the design rationale.


## Parameters

Single tagged `parameters` cell. The default `INPUT_CSV` points at the
*new* short path Notebook 02 writes
(`sheet_<x>/data/derived.csv`), not the old `<stem>__0__derived.csv`.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Required. These are normally supplied by run_pipeline.sh through
# Papermill. Keeping them as None prevents accidental runs against old
# local files.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) -------------------------------------
# Path to a .json/.yml/.yaml file that deep-merges onto DEFAULT_CONFIG.
# None, "None", "null", or "" all mean use DEFAULT_CONFIG.
CONFIG_PATH = None

# --- Stage 1A behaviour --------------------------------------------
# Comma-separated override list for the duplicate-key tuple, e.g.
# "sample_id,replicate_id,sample_date,station_id". None -> auto-pick from
# config["duplicate_key_candidates"].
DUPLICATE_KEYS = None

# If True, copy resolved aliases into canonical columns and keep the
# original source columns too. This is useful while auditing the pipeline.
PRESERVE_ORIGINAL_COLUMNS = True

# If True, skip Parquet writes and write CSV only.
NO_PARQUET = False

# Useful during development: prepare everything but write nothing.
DRY_RUN = False

# If True, print large tables in console form. If False, use rich display().
PRINT_COLUMNS = False


## Setup

The Stage itself lives in `roap.stage1a`: `run(df, stage_config)` orders the
`roap.schema` steps (canonical schema, range, presence and duplicate flags)
and returns a `StageResult` without touching disk. This notebook reads the
input, calls `run`, writes the result and shows the tables.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import stage1a
from roap.common import as_bool, die, is_none_like
from roap.stage_config import load_stage_config
from roap.stage_result import write_stage_result

## Load input CSV and config

In [ ]:
PRESERVE_ORIGINAL_COLUMNS = as_bool(PRESERVE_ORIGINAL_COLUMNS)
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
PRINT_COLUMNS = as_bool(PRINT_COLUMNS)

if is_none_like(INPUT_CSV):
    die("INPUT_CSV is required. Run this notebook through run_pipeline.sh or set INPUT_CSV.")
if is_none_like(OUT_DIR):
    die("OUT_DIR is required. Run this notebook through run_pipeline.sh or set OUT_DIR.")
config_path = None if is_none_like(CONFIG_PATH) else str(CONFIG_PATH).strip()

src_csv = Path(str(INPUT_CSV).strip()).expanduser().resolve()
if not src_csv.exists():
    die(f"File not found: {src_csv}\nDid Notebook 02 run successfully? Stage 1A reads its derived.csv.")
if src_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {src_csv.name}")
out_root = Path(str(OUT_DIR).strip()).expanduser().resolve()

# Stage 1A defaults <- this Stage's optional config file (#41).
stage_cfg = load_stage_config("stage1a", config_path)
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")

dup_override = None
if not is_none_like(DUPLICATE_KEYS):
    dup_override = [x.strip() for x in str(DUPLICATE_KEYS).split(",") if x.strip()] or None

df_in = pd.read_csv(src_csv)
print(f"Source CSV    : {src_csv}")
print(f"Rows          : {len(df_in):,}")
print(f"Config source : {stage_cfg.source or 'DEFAULT_CONFIG'}")

## Run Stage 1A

`stage1a.run` resolves each canonical column from its aliases, normalises
TA-unit and pH-scale labels, then adds three classes of advisory flag (a
flag never deletes a row):

- **Range flags** (`flag_*_out_of_range`) with bounds from the Stage's
  `range_policy`.
- **Presence flags**: `flag_required_core_missing`, `flag_ta_units_*`,
  `flag_ph_scale_observed_missing`, `flag_pressure_output_dbar_missing`.
- **Duplicate flag** (`flag_possible_duplicate`) on the first key tuple in
  `duplicate_key_candidates` whose columns all exist and hold a value, or
  on `DUPLICATE_KEYS` when set.

It returns `data/staged.csv` (the Stage 1B input), `data/analysis_ready.csv`
(canonical export order), the audit tables, the report and the manifest.

In [ ]:
try:
    result = stage1a.run(
        df_in,
        stage_cfg,
        source=str(src_csv),
        duplicate_keys=dup_override,
        preserve_original_columns=PRESERVE_ORIGINAL_COLUMNS,
    )
except ValueError as exc:
    die(str(exc))

result.manifest["output_root"] = str(out_root)
result.manifest["parameters"].update(
    {
        "INPUT_CSV": str(src_csv),
        "OUT_DIR": str(out_root),
        "CONFIG_PATH": config_path,
        "DUPLICATE_KEYS": DUPLICATE_KEYS,
        "NO_PARQUET": NO_PARQUET,
        "DRY_RUN": DRY_RUN,
        "PRINT_COLUMNS": PRINT_COLUMNS,
    }
)
staged_df = result.data["data/staged.csv"]
analysis_ready_df = result.data["data/analysis_ready.csv"]
rename_audit = result.tables["logs/rename_audit.csv"]
canonical_inventory = result.tables["logs/canonical_inventory.csv"]
missing_tbl = result.tables["logs/missingness.csv"]
print(f"Canonical actions applied : {len(rename_audit)}")
print(f"Duplicate keys used       : {result.manifest['policy']['duplicate_keys_used']}")
print(f"Rows flagged as duplicate : {result.manifest['row_counts']['possible_duplicate_rows']}")
print(f"Canonical columns present : {int(canonical_inventory['present'].sum())} / {len(canonical_inventory)}")

## Write outputs

```
<OUT_DIR>/
    data/staged.csv, analysis_ready.csv      (and .parquet)
    reports/report.md
    logs/manifest.json, effective_config.json, source_lookup.json,
         rename_audit.csv, canonical_inventory.csv, missingness.csv
    run_bundle_manifest.json
```

`DRY_RUN = True` skips the writes. `NO_PARQUET = True` writes CSV only.
The run bundle hashes the input, the config and every file this run
wrote, so `roap verify-bundle <OUT_DIR>` can check them later.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    written = write_stage_result(result, out_root, parquet=not NO_PARQUET)
    print(f"Stage 1A complete: {len(written)} files under {out_root}")
    print(f"  -> Stage 1B input: {written['data/staged.csv']}")

## Review outputs

In [ ]:
if PRINT_COLUMNS or display is None:
    print("=== Missingness (top 15) ===")
    print(missing_tbl.head(15).to_string(index=False))
    print("\n=== Rename audit ===")
    print(rename_audit.to_string(index=False))
    print("\n=== Canonical inventory (first 20) ===")
    print(canonical_inventory.head(20).to_string(index=False))
else:
    display(missing_tbl.head(15))
    display(rename_audit)
    display(canonical_inventory.head(20))
    display(analysis_ready_df.head(10))